# Exam (Elite) — Practice Problems

Ceiling problems. Only attempt after MEDIUM and HARD feel comfortable. If you can solve these cold in 30 min, you’re overqualified.

**Mix:** 40% data pipelines / 20% ML / 40% stretch modeling

**Rules:**
- 30-minute timer per problem
- No docs, no autocomplete, no outside help
- Run the test cell — all assertions must pass

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import numpy as np
from typing import Any, Iterator
from collections import defaultdict, Counter, OrderedDict
import math
import heapq

---

## Problem 1: Full Embedding Pipeline

### Scenario

You’re building a complete data curation system for a video dataset. Given pre-computed embeddings for each item, you need to: store and manage them, detect near-duplicates, and then select a maximally diverse subset.

### Formulas

**Cosine similarity:** `cos(a, b) = (a · b) / (||a|| * ||b||)`

**Connected components:** Two items are in the same cluster if there exists a path of near-duplicate pairs between them. Use BFS or union-find.

**Farthest-first traversal (diversity selection):**
1. Start with the item that has the highest L2 norm (arbitrary tiebreaker)
2. At each step, add the item whose minimum cosine distance to any already-selected item is maximized
3. Cosine distance = 1 - cosine_similarity

### Requirements

**`EmbeddingStore`:**
- `__init__(self)` — empty store
- `add(self, item_id: str, embedding: torch.Tensor, metadata: dict | None = None)` — add an item. Raise ValueError if item_id already exists.
- `remove(self, item_id: str)` — remove an item. Raise KeyError if not found.
- `update(self, item_id: str, embedding: torch.Tensor | None = None, metadata: dict | None = None)` — update embedding and/or metadata. Raise KeyError if not found.
- `get(self, item_id: str) -> dict` — return `{"embedding": tensor, "metadata": dict}`
- `get_all_embeddings(self) -> tuple[list[str], torch.Tensor]` — return (list of IDs, stacked embeddings matrix (N, D))
- `__len__(self)` — number of stored items

**`DuplicateDetector`:**
- `__init__(self, store: EmbeddingStore, threshold: float)`
- `find_clusters(self) -> list[set[str]]` — find connected components of items with pairwise cosine similarity >= threshold. Return only clusters with 2+ items.

**`DiversitySelector`:**
- `__init__(self, store: EmbeddingStore)`
- `select(self, candidate_ids: list[str], target_count: int) -> list[str]` — farthest-first traversal on the given candidate IDs. Return up to target_count IDs. If target_count >= len(candidate_ids), return all.

**`run_pipeline(store: EmbeddingStore, dup_threshold: float, target_count: int) -> dict`:**
- Detect duplicate clusters
- For each cluster, keep only the item with the highest L2 norm embedding (arbitrary tiebreaker for ties)
- From the remaining (kept + non-duplicated) items, run diversity selection to pick target_count items
- Return `{"selected_ids": list[str], "duplicate_clusters": list[set[str]], "removed_as_duplicates": list[str], "total_original": int, "total_after_dedup": int, "total_selected": int}`

In [ ]:
class EmbeddingStore:
    # YOUR CODE HERE
    pass


class DuplicateDetector:
    # YOUR CODE HERE
    pass


class DiversitySelector:
    # YOUR CODE HERE
    pass


def run_pipeline(store: EmbeddingStore, dup_threshold: float, target_count: int) -> dict:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 1 ---
torch.manual_seed(42)

# === EmbeddingStore basic operations ===
store = EmbeddingStore()
store.add("a", torch.randn(16), {"source": "web"})
store.add("b", torch.randn(16), {"source": "scraped"})
assert len(store) == 2

# Duplicate ID should raise
try:
    store.add("a", torch.randn(16))
    assert False, "Should raise ValueError for duplicate ID"
except ValueError:
    pass

# Get
item = store.get("a")
assert item["embedding"].shape == (16,)
assert item["metadata"]["source"] == "web"

# Update
new_emb = torch.ones(16)
store.update("a", embedding=new_emb)
assert torch.allclose(store.get("a")["embedding"], new_emb)
store.update("a", metadata={"source": "updated"})
assert store.get("a")["metadata"]["source"] == "updated"

# Remove
store.remove("b")
assert len(store) == 1
try:
    store.remove("nonexistent")
    assert False, "Should raise KeyError"
except KeyError:
    pass

# get_all_embeddings
store.add("b", torch.randn(16))
store.add("c", torch.randn(16))
ids, emb_matrix = store.get_all_embeddings()
assert len(ids) == 3
assert emb_matrix.shape == (3, 16)

# === DuplicateDetector ===
torch.manual_seed(42)
dup_store = EmbeddingStore()
base_a = F.normalize(torch.randn(32), dim=0)
base_b = F.normalize(torch.randn(32), dim=0)

# Cluster A: items 0, 1, 2 are near-duplicates
dup_store.add("item_0", base_a + 0.01 * torch.randn(32))
dup_store.add("item_1", base_a + 0.01 * torch.randn(32))
dup_store.add("item_2", base_a + 0.01 * torch.randn(32))
# Cluster B: items 3, 4
dup_store.add("item_3", base_b + 0.01 * torch.randn(32))
dup_store.add("item_4", base_b + 0.01 * torch.randn(32))
# Unique item
dup_store.add("item_5", F.normalize(torch.randn(32), dim=0))

detector = DuplicateDetector(dup_store, threshold=0.95)
clusters = detector.find_clusters()
assert len(clusters) == 2, f"Expected 2 clusters, got {len(clusters)}"

# Find which cluster contains item_0
cluster_a = [c for c in clusters if "item_0" in c][0]
cluster_b = [c for c in clusters if "item_3" in c][0]
assert cluster_a == {"item_0", "item_1", "item_2"}
assert cluster_b == {"item_3", "item_4"}
assert not any("item_5" in c for c in clusters), "Unique item should not be in any cluster"

# === DiversitySelector ===
torch.manual_seed(42)
div_store = EmbeddingStore()
# Place items at known positions for predictable diversity selection
div_store.add("north", torch.tensor([0.0, 1.0, 0.0, 0.0]))
div_store.add("south", torch.tensor([0.0, -1.0, 0.0, 0.0]))
div_store.add("east", torch.tensor([1.0, 0.0, 0.0, 0.0]))
div_store.add("west", torch.tensor([-1.0, 0.0, 0.0, 0.0]))
div_store.add("near_north", torch.tensor([0.0, 0.99, 0.0, 0.0]))

selector = DiversitySelector(div_store)
selected = selector.select(["north", "south", "east", "west", "near_north"], target_count=3)
assert len(selected) == 3
# near_north is very close to north, so it should NOT be among the first 3 selected
assert "near_north" not in selected, f"near_north should not be selected early: {selected}"
# All 4 cardinal directions should be preferred over near_north
selected_4 = selector.select(["north", "south", "east", "west", "near_north"], target_count=4)
assert "near_north" not in selected_4, f"near_north should still not be in top 4: {selected_4}"

# target_count >= candidates returns all
selected_all = selector.select(["north", "south"], target_count=10)
assert len(selected_all) == 2

# === Full pipeline integration ===
result = run_pipeline(dup_store, dup_threshold=0.95, target_count=3)
assert result["total_original"] == 6
assert len(result["duplicate_clusters"]) == 2
assert len(result["removed_as_duplicates"]) > 0
assert result["total_after_dedup"] == result["total_original"] - len(result["removed_as_duplicates"])
assert result["total_selected"] == 3
assert len(result["selected_ids"]) == 3
# Selected IDs should not include any removed duplicates
removed_set = set(result["removed_as_duplicates"])
for sid in result["selected_ids"]:
    assert sid not in removed_set, f"Selected ID {sid} was removed as duplicate"

print("Problem 1: ALL TESTS PASSED")

---

## Problem 2: Curriculum Training Scheduler

### Scenario

Research shows that training on easy examples first and gradually introducing harder ones (curriculum learning) improves convergence. Build a scheduler that manages this.

### Formulas

**Pacing function:** At training progress `p` (0.0 to 1.0), include data up to difficulty percentile:
```
max_percentile(p) = min(1.0, start_percentile + p * (1.0 - start_percentile))
```
Where `start_percentile` is a hyperparameter (e.g., 0.2 means start with the easiest 20%).

So at `p=0`: include data up to percentile `start_percentile` (e.g., easiest 20%).
At `p=1`: include everything (`percentile=1.0`).

### Requirements

**`DifficultyScorer`:**
- `__init__(self, rules: list[tuple[str, callable]])` — each rule is `(field_name, scoring_fn)`. `scoring_fn` takes a field value and returns a float score. Missing fields contribute 0.
- `score(self, record: dict) -> float` — sum of all rule scores for this record
- `score_batch(self, records: list[dict]) -> list[float]` — scores for all records

**`CurriculumScheduler`:**
- `__init__(self, records: list[dict], scorer: DifficultyScorer, start_percentile: float = 0.2)`
- Internally, score all records and sort by difficulty
- `get_available_indices(self, progress: float) -> list[int]` — return indices of records whose difficulty score is at or below the max_percentile(progress) cutoff. Uses the original indices (position in the input list).
- `sample_batch(self, batch_size: int, progress: float) -> list[int]` — randomly sample batch_size indices from available records at this progress. If batch_size > available count, sample with replacement.
- `get_difficulty_stats(self) -> dict` — return `{"min": float, "max": float, "mean": float, "median": float, "scores": list[float]}` where scores is in original record order

In [ ]:
class DifficultyScorer:
    # YOUR CODE HERE
    pass


class CurriculumScheduler:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 2 ---
torch.manual_seed(42)
np.random.seed(42)

# === DifficultyScorer ===
scorer = DifficultyScorer([
    ("caption_length", lambda x: x / 100.0),  # longer caption = harder
    ("resolution", lambda x: x / 1024.0),      # higher res = harder
])

# Basic scoring
easy_record = {"caption_length": 10, "resolution": 128}
hard_record = {"caption_length": 90, "resolution": 1024}
score_easy = scorer.score(easy_record)
score_hard = scorer.score(hard_record)
assert score_easy < score_hard, f"Easy ({score_easy}) should be less than hard ({score_hard})"

# Missing field contributes 0
partial_record = {"caption_length": 50}
score_partial = scorer.score(partial_record)
assert abs(score_partial - 50/100.0) < 1e-5, f"Expected 0.5, got {score_partial}"

# Batch scoring
batch_scores = scorer.score_batch([easy_record, hard_record, partial_record])
assert len(batch_scores) == 3
assert abs(batch_scores[0] - score_easy) < 1e-5
assert abs(batch_scores[1] - score_hard) < 1e-5

# === CurriculumScheduler ===
# Create records with clear difficulty tiers
records = []
for i in range(100):
    records.append({
        "caption_length": i,          # 0 to 99
        "resolution": 128 + i * 9,    # 128 to 1019
    })

scheduler = CurriculumScheduler(records, scorer, start_percentile=0.2)

# At progress=0.0, only easiest 20% should be available
avail_start = scheduler.get_available_indices(progress=0.0)
assert len(avail_start) == 20, f"Expected 20 available at start, got {len(avail_start)}"
# These should be the records with the lowest difficulty scores (indices 0-19)
assert set(avail_start) == set(range(20)), f"Expected indices 0-19, got {sorted(avail_start)}"

# At progress=0.5, should have 60% available
# max_percentile = min(1.0, 0.2 + 0.5 * 0.8) = min(1.0, 0.6) = 0.6
avail_mid = scheduler.get_available_indices(progress=0.5)
assert len(avail_mid) == 60, f"Expected 60 available at p=0.5, got {len(avail_mid)}"

# At progress=1.0, everything should be available
avail_end = scheduler.get_available_indices(progress=1.0)
assert len(avail_end) == 100, f"Expected 100 available at end, got {len(avail_end)}"

# Monotonicity: more progress → more data
for p in [0.0, 0.25, 0.5, 0.75, 1.0]:
    avail = scheduler.get_available_indices(p)
    if p > 0:
        assert len(avail) >= prev_count, "Available data should increase with progress"
    prev_count = len(avail)

# sample_batch at progress=0 should only return easy items
batch_early = scheduler.sample_batch(batch_size=10, progress=0.0)
assert len(batch_early) == 10
assert all(idx < 20 for idx in batch_early), f"Early batch should only have easy items: {batch_early}"

# sample_batch with replacement when batch_size > available
big_batch = scheduler.sample_batch(batch_size=50, progress=0.0)
assert len(big_batch) == 50  # only 20 available, so must sample with replacement
assert all(idx < 20 for idx in big_batch)

# Difficulty stats
stats = scheduler.get_difficulty_stats()
assert stats["min"] < stats["max"]
assert abs(stats["mean"] - np.mean(stats["scores"])) < 1e-5
assert len(stats["scores"]) == 100
# First record (easiest) should have lowest score
assert stats["scores"][0] < stats["scores"][99]

print("Problem 2: ALL TESTS PASSED")

---

## Problem 3: Denoising Autoencoder + Data Inspection

**Scenario:** Build a denoising autoencoder that learns to reconstruct clean data from noisy inputs, then use it for data quality inspection — identifying outliers and corrupted samples.

**Spec:**

**`DenoisingAutoencoder(nn.Module)`:**
- `__init__(self, data_dim: int, hidden_dim: int = 64, latent_dim: int = 16)`
- **Encoder:** Linear(data_dim, hidden_dim) → ReLU → Linear(hidden_dim, latent_dim)
- **Decoder:** Linear(latent_dim, hidden_dim) → ReLU → Linear(hidden_dim, data_dim)
- `encode(self, x)` — return latent representation
- `decode(self, z)` — return reconstruction
- `forward(self, x)` — encode then decode
- `add_noise(self, x, noise_factor=0.3)` — add Gaussian noise: `x + noise_factor * torch.randn_like(x)`

**Training function:**
`train_dae(model, data, n_epochs=100, lr=1e-3, noise_factor=0.3) -> list[float]`
- Train with MSE loss between clean input and reconstruction of noisy input
- Return list of per-epoch average losses

**Data inspection:**
`inspect_data(model, data, threshold_percentile=95) -> dict`
- Compute reconstruction error (MSE per sample) for each data point
- Identify outliers: samples whose reconstruction error exceeds the threshold_percentile of errors
- Return:
  - `"errors"`: (N,) tensor of per-sample reconstruction errors
  - `"mean_error"`: float
  - `"outlier_indices"`: list of ints (indices of outlier samples)
  - `"outlier_threshold"`: float (the percentile threshold value)
  - `"latent_representations"`: (N, latent_dim) tensor from the encoder

**Constraints:**
- Use standard PyTorch nn.Module and autograd for training.

In [ ]:
class DenoisingAutoencoder(nn.Module):
    """Denoising autoencoder for data quality inspection.

    Args:
        data_dim: Input/output dimension.
        hidden_dim: Hidden layer size.
        latent_dim: Latent space dimension.
    """

    def __init__(self, data_dim: int, hidden_dim: int = 64, latent_dim: int = 16):
        super().__init__()
        # YOUR CODE HERE
        pass

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def add_noise(self, x: torch.Tensor, noise_factor: float = 0.3) -> torch.Tensor:
        # YOUR CODE HERE
        pass


def train_dae(
    model: DenoisingAutoencoder,
    data: torch.Tensor,
    n_epochs: int = 100,
    lr: float = 1e-3,
    noise_factor: float = 0.3,
) -> list[float]:
    """Train denoising autoencoder. Returns per-epoch losses."""
    # YOUR CODE HERE
    pass


def inspect_data(
    model: DenoisingAutoencoder,
    data: torch.Tensor,
    threshold_percentile: float = 95,
) -> dict:
    """Inspect data quality using trained DAE."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 3 ---
torch.manual_seed(42)

data_dim = 20
N = 500

# Generate structured data (not random noise — DAE should learn patterns)
# 3 clusters in 20D space
clean_data = torch.cat([
    torch.randn(200, data_dim) * 0.5 + torch.randn(1, data_dim) * 3,
    torch.randn(200, data_dim) * 0.5 + torch.randn(1, data_dim) * 3,
    torch.randn(100, data_dim) * 0.5 + torch.randn(1, data_dim) * 3,
])

# Add some outliers (random high-magnitude noise)
outlier_data = torch.randn(10, data_dim) * 10
data = torch.cat([clean_data, outlier_data])  # 510 total, last 10 are outliers

# Model
dae = DenoisingAutoencoder(data_dim=data_dim, hidden_dim=64, latent_dim=8)

# Shape checks
x = torch.randn(4, data_dim)
z = dae.encode(x)
assert z.shape == (4, 8), f"Latent shape: expected (4,8), got {z.shape}"
x_recon = dae.decode(z)
assert x_recon.shape == (4, data_dim), f"Recon shape: expected (4,{data_dim}), got {x_recon.shape}"
x_full = dae(x)
assert x_full.shape == x.shape

# Noise addition
x_noisy = dae.add_noise(x, noise_factor=0.3)
assert x_noisy.shape == x.shape
assert not torch.allclose(x, x_noisy), "Noisy should differ from clean"

# Training
torch.manual_seed(42)
dae = DenoisingAutoencoder(data_dim=data_dim, hidden_dim=64, latent_dim=8)
losses = train_dae(dae, data[:500], n_epochs=50, lr=1e-3, noise_factor=0.3)
assert len(losses) == 50
assert losses[-1] < losses[0], f"Loss should decrease: {losses[0]:.4f} -> {losses[-1]:.4f}"

# Inspection
dae.eval()
result = inspect_data(dae, data, threshold_percentile=95)

# Shape checks
assert result["errors"].shape == (510,)
assert isinstance(result["mean_error"], float)
assert result["mean_error"] > 0
assert isinstance(result["outlier_indices"], list)
assert result["latent_representations"].shape == (510, 8)

# Outlier threshold
assert result["outlier_threshold"] > 0

# Outliers should have higher reconstruction error than most clean samples
clean_errors = result["errors"][:500].mean().item()
outlier_errors = result["errors"][500:].mean().item()
assert outlier_errors > clean_errors, \
    f"Outliers should have higher error: clean={clean_errors:.4f}, outlier={outlier_errors:.4f}"

# At least some of the known outliers should be detected
detected = set(result["outlier_indices"])
known_outliers = set(range(500, 510))
overlap = detected & known_outliers
assert len(overlap) >= 3, \
    f"Should detect at least 3/10 outliers, detected {len(overlap)}: {overlap}"

print("Problem 3: ALL TESTS PASSED")

---

## Problem 4: Flow Matching (Forward Process + Euler Sampling)

**Scenario:** Flow matching is a modern alternative to DDPM diffusion. Instead of learning to reverse a noise process, it learns a velocity field that transforms noise into data along a straight path.

**Spec:**

**Interpolation path (Optimal Transport):**
$$x_t = (1 - t) \cdot x_0 + t \cdot \epsilon$$

where $x_0$ is clean data, $\epsilon \sim \mathcal{N}(0, I)$, and $t \in [0, 1]$.

**Target velocity field:**
$$u_t(x_t | x_0) = \epsilon - x_0$$

This is the constant velocity that moves $x_0$ to $\epsilon$ in a straight line.

**Training objective:**
$$L = \mathbb{E}_{t, x_0, \epsilon} \| v_\theta(x_t, t) - u_t \|^2$$

**Sampling (Euler method):**
Starting from $x_1 \sim \mathcal{N}(0, I)$, integrate backwards:
$$x_{t - \Delta t} = x_t - \Delta t \cdot v_\theta(x_t, t)$$

with steps at $t = 1, 1-\Delta t, 1-2\Delta t, \ldots, \Delta t$ where $\Delta t = 1/N_{steps}$.

**`FlowMatchingSchedule`:**
- `__init__(self)`
- `sample_timesteps(self, batch_size: int) -> torch.Tensor` — sample uniform t in [0, 1], shape (B,)
- `interpolate(self, x_0: torch.Tensor, noise: torch.Tensor, t: torch.Tensor) -> torch.Tensor` — compute $x_t$
- `compute_target(self, x_0: torch.Tensor, noise: torch.Tensor) -> torch.Tensor` — compute target velocity $u_t = \epsilon - x_0$
- `compute_loss(self, model_output: torch.Tensor, target: torch.Tensor) -> torch.Tensor` — MSE loss

**`VelocityNetwork(nn.Module)`:**
- `__init__(self, data_dim: int, hidden_dim: int = 128, time_dim: int = 32)`
- Embeds time $t$ using sinusoidal embedding (dim = time_dim)
- MLP: concat(x, time_embed) → Linear → SiLU → Linear → SiLU → Linear → output
- `forward(self, x, t)` — x: (B, data_dim), t: (B,) → (B, data_dim)

**`euler_sample(model, schedule, shape, num_steps=100) -> torch.Tensor`:**
- Start from $x_1 \sim \mathcal{N}(0, I)$
- Iterate: $t = 1, 1-dt, 1-2dt, \ldots, dt$ where $dt = 1/\text{num\_steps}$
- At each step: $x_{t-dt} = x_t - dt \cdot v_\theta(x_t, t)$
- Return final $x_0$

**Constraints:**
- Sinusoidal time embedding (same formula as standard positional encoding).

In [ ]:
class FlowMatchingSchedule:
    """Flow matching schedule with optimal transport interpolation."""

    def sample_timesteps(self, batch_size: int) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def interpolate(self, x_0: torch.Tensor, noise: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def compute_target(self, x_0: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def compute_loss(self, model_output: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass


class SinusoidalTimeEmbedding(nn.Module):
    """Sinusoidal embedding for scalar timestep."""

    def __init__(self, dim: int):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass


class VelocityNetwork(nn.Module):
    """MLP that predicts velocity field v(x, t).

    Args:
        data_dim: Dimension of data.
        hidden_dim: Hidden layer size.
        time_dim: Time embedding dimension.
    """

    def __init__(self, data_dim: int, hidden_dim: int = 128, time_dim: int = 32):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass


def euler_sample(
    model: VelocityNetwork,
    schedule: FlowMatchingSchedule,
    shape: tuple,
    num_steps: int = 100,
) -> torch.Tensor:
    """Generate samples using Euler integration of the learned velocity field."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Problem 4 ---
torch.manual_seed(42)

schedule = FlowMatchingSchedule()

# Timestep sampling
t = schedule.sample_timesteps(100)
assert t.shape == (100,)
assert (t >= 0).all() and (t <= 1).all(), "Timesteps should be in [0, 1]"

# Interpolation
x_0 = torch.randn(4, 8)
noise = torch.randn(4, 8)
t_batch = torch.tensor([0.0, 0.5, 1.0, 0.3])

x_t = schedule.interpolate(x_0, noise, t_batch)
assert x_t.shape == (4, 8)

# At t=0: x_t should be x_0
assert torch.allclose(x_t[0], x_0[0], atol=1e-5), "At t=0, x_t should be x_0"
# At t=1: x_t should be noise
assert torch.allclose(x_t[2], noise[2], atol=1e-5), "At t=1, x_t should be noise"
# At t=0.5: x_t should be midpoint
midpoint = 0.5 * x_0[1] + 0.5 * noise[1]
assert torch.allclose(x_t[1], midpoint, atol=1e-5), "At t=0.5, x_t should be midpoint"

# Target velocity
target = schedule.compute_target(x_0, noise)
assert target.shape == (4, 8)
assert torch.allclose(target, noise - x_0, atol=1e-5), "Target should be noise - x_0"

# Loss
pred = torch.randn(4, 8)
loss = schedule.compute_loss(pred, target)
assert loss.ndim == 0, "Loss should be scalar"
assert torch.allclose(loss, F.mse_loss(pred, target))

# VelocityNetwork
vnet = VelocityNetwork(data_dim=8, hidden_dim=64, time_dim=16)
v_pred = vnet(x_0, t_batch)
assert v_pred.shape == (4, 8), f"Expected (4,8), got {v_pred.shape}"

# Different timesteps should produce different outputs
v1 = vnet(x_0[:1].expand(2, -1), torch.tensor([0.1, 0.9]))
assert not torch.allclose(v1[0], v1[1], atol=1e-3), "Different t should give different output"

# Gradient flow
loss_g = schedule.compute_loss(vnet(x_0, t_batch), target)
loss_g.backward()
has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in vnet.parameters())
assert has_grad, "Gradients should flow through velocity network"

# Euler sampling
torch.manual_seed(42)
vnet_sample = VelocityNetwork(data_dim=4, hidden_dim=32, time_dim=16)
samples = euler_sample(vnet_sample, schedule, shape=(8, 4), num_steps=10)
assert samples.shape == (8, 4), f"Expected (8,4), got {samples.shape}"
assert samples.dtype == torch.float32

# Deterministic with same seed
torch.manual_seed(42)
samples2 = euler_sample(vnet_sample, schedule, shape=(8, 4), num_steps=10)
assert torch.allclose(samples, samples2), "Same seed should give same samples"

# Different seed gives different samples
torch.manual_seed(99)
samples3 = euler_sample(vnet_sample, schedule, shape=(8, 4), num_steps=10)
assert not torch.allclose(samples, samples3), "Different seed should give different samples"

# Training sanity check: loss should decrease
torch.manual_seed(42)
vnet_train = VelocityNetwork(data_dim=4, hidden_dim=32, time_dim=16)
optimizer = torch.optim.Adam(vnet_train.parameters(), lr=1e-3)
train_data = torch.randn(64, 4)  # simple random data

loss_start = None
for step in range(200):
    optimizer.zero_grad()
    t_step = schedule.sample_timesteps(64)
    eps = torch.randn_like(train_data)
    x_t_step = schedule.interpolate(train_data, eps, t_step)
    target_step = schedule.compute_target(train_data, eps)
    pred_step = vnet_train(x_t_step, t_step)
    loss_step = schedule.compute_loss(pred_step, target_step)
    loss_step.backward()
    optimizer.step()
    if loss_start is None:
        loss_start = loss_step.item()

loss_end = loss_step.item()
assert loss_end < loss_start, f"Training should reduce loss: {loss_start:.4f} -> {loss_end:.4f}"

print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: LoRA (Low-Rank Adaptation)

### Scenario

Fine-tuning large models is expensive. LoRA adds trainable low-rank matrices to frozen weights, reducing trainable parameters by 100x+.

### Formulas

**Standard linear:** `y = x @ W^T + b` where W is (out_features, in_features)

**LoRA forward:**
```
y = x @ W^T + b + (x @ A^T @ B^T) * scale
```
- A: (rank, in_features) — initialized from N(0, 1)
- B: (out_features, rank) — initialized to zeros
- scale: alpha / rank (a constant)
- W and b are **frozen** (require_grad=False)
- Only A and B are trainable

**Merge:** After training, fold LoRA into the base weight:
```
W_merged = W + (B @ A) * scale
```
Result is a single nn.Linear with no LoRA overhead.

### Requirements

**`LoRALinear(nn.Module)`:**
- `__init__(self, base_linear: nn.Linear, rank: int, alpha: float = 1.0)`
- Freeze base_linear weights and bias
- Create trainable A (rank, in_features) and B (out_features, rank)
- A initialized with `nn.init.kaiming_uniform_`, B initialized to zeros
- `forward(self, x)` — compute original + low-rank delta
- `merge(self) -> nn.Linear` — return a new nn.Linear with merged weights

**`inject_lora(model: nn.Module, rank: int, alpha: float = 1.0) -> nn.Module`:**
- Replace ALL nn.Linear layers in the model with LoRALinear wrappers
- Handle nested modules (use named_modules + setattr)
- Return the modified model

**`merge_lora(model: nn.Module) -> nn.Module`:**
- Replace ALL LoRALinear layers with their merged nn.Linear equivalents
- Return the modified model

In [ ]:
class LoRALinear(nn.Module):
    # YOUR CODE HERE
    pass


def inject_lora(model: nn.Module, rank: int, alpha: float = 1.0) -> nn.Module:
    # YOUR CODE HERE
    pass


def merge_lora(model: nn.Module) -> nn.Module:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 5 ---
torch.manual_seed(42)

# Basic LoRALinear
base = nn.Linear(64, 32)
lora = LoRALinear(base, rank=4, alpha=1.0)
x = torch.randn(2, 64)
out = lora(x)
assert out.shape == (2, 32), f"Expected (2,32), got {out.shape}"

# B initialized to zeros → LoRA delta is zero at init → output matches base
base_out = base(x)
assert torch.allclose(out, base_out, atol=1e-5), "At init (B=0), LoRA output should match base"

# Base weights are frozen
for p in [lora.base_linear.weight, lora.base_linear.bias]:
    assert not p.requires_grad, "Base weights should be frozen"

# LoRA weights are trainable
assert lora.A.requires_grad, "A should be trainable"
assert lora.B.requires_grad, "B should be trainable"

# Trainable param count is much smaller
total_base = sum(p.numel() for p in base.parameters())
trainable_lora = sum(p.numel() for p in lora.parameters() if p.requires_grad)
assert trainable_lora < total_base, f"LoRA params ({trainable_lora}) should be < base ({total_base})"

# Training changes LoRA output
optimizer = torch.optim.SGD([p for p in lora.parameters() if p.requires_grad], lr=0.1)
target = torch.randn(2, 32)
loss = F.mse_loss(lora(x), target)
loss.backward()
optimizer.step()
out_after = lora(x)
assert not torch.allclose(out_after, base_out, atol=1e-5), "After training, LoRA output should differ"

# Merge produces equivalent nn.Linear
merged = lora.merge()
assert isinstance(merged, nn.Linear), "merge() should return nn.Linear"
merged_out = merged(x)
lora_out = lora(x)
assert torch.allclose(merged_out, lora_out, atol=1e-5), "Merged output should match LoRA output"

# inject_lora replaces all Linear layers
model = nn.Sequential(
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 32),
)
model_lora = inject_lora(model, rank=4)
num_lora = sum(1 for m in model_lora.modules() if isinstance(m, LoRALinear))
assert num_lora == 2, f"Expected 2 LoRALinear, got {num_lora}"
num_plain_linear = sum(1 for m in model_lora.modules() if type(m) == nn.Linear)
# The base_linear inside LoRALinear are nn.Linear but not direct children of Sequential
# The Sequential should have LoRALinear, not plain nn.Linear
assert isinstance(model_lora[0], LoRALinear), "First layer should be LoRALinear"
assert isinstance(model_lora[2], LoRALinear), "Third layer should be LoRALinear"

# merge_lora restores plain nn.Linear
x_test = torch.randn(3, 64)
out_before_merge = model_lora(x_test)
model_merged = merge_lora(model_lora)
num_lora_after = sum(1 for m in model_merged.modules() if isinstance(m, LoRALinear))
assert num_lora_after == 0, "After merge, no LoRALinear should remain"
out_after_merge = model_merged(x_test)
assert torch.allclose(out_before_merge, out_after_merge, atol=1e-5), "Merged model should give same output"

print("Problem 5: ALL TESTS PASSED")

---

## Scoring

| Result | Assessment |
|--------|------------|
| Solved in < 25 min, all tests pass | **Overqualified** — you’re ready for anything |
| Solved in 25-30 min, minor debugging | **Strong pass** |
| 30-40 min or needed hints | **Pass** — you can handle hard problems |
| Couldn’t solve or > 40 min | **Go back to HARD** — nail those first |